In [1]:
import os
import csv
from osgeo import gdal
import numpy as np
import re
import itertools

In [15]:
def get_raster_file_list(path):
    """Get a list of the raster files inside the folder"""
    File_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
    for file in os.listdir(path):
        if file.endswith(".tif") or file.endswith(".tiff"):
            if file not in File_list:
                File_list.append(os.path.join(path,file))
        else:
            pass
    return File_list

def read_csv_first_column(csv_path):
    """
    Reads a CSV file and extracts the first column as landcover class values.
    Skips the header row.

    Returns: list of integers
    """
    try:
        classes = []
        with open(csv_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.reader(f)
            # next(reader)  # skip header
            for row in reader:
                if row and row[0].strip() != "":
                    try:
                        classes.append(int(row[0]))
                    except ValueError:
                        pass  # ignore non-numeric values
        return classes
    except:
        return None

def match_rasters_by_country(data_rasters, landcover_rasters):
    # Extract country = first token before underscore
    def country(f): 
        return os.path.basename(f).split("_")[0]

    # Build lookup tables
    data_dict = {}
    for f in data_rasters:
        data_dict.setdefault(country(f), []).append(f)

    lc_dict = {}
    for f in landcover_rasters:
        lc_dict.setdefault(country(f), []).append(f)

    # Match only countries appearing in both lists
    matched_pairs = []
    for c in set(data_dict) & set(lc_dict):
        for d, l in itertools.product(data_dict[c], lc_dict[c]):
            matched_pairs.append((d, l))

    return matched_pairs

def match_all_rasters(data_rasters, landcover_rasters):
    return list(itertools.product(data_rasters, landcover_rasters))

def match_rasters_by_country_year(data_rasters, landcover_rasters):
    """
    Matches data rasters and landcover rasters using:
        - the first string in the filename (country)
        - the last 4-digit number in the filename (year)

    Expected filename examples:
        BRA_carbon_2000.tif
        BRA_landcover_2000.tif
        USA_vsc_2015_v1.tif
        USA_landcover_2015.tif

    Returns:
        list of (data_raster, landcover_raster) pairs
    """

    def extract_country_and_year(filepath):
        fname = os.path.basename(filepath)

        # country = first token before first underscore
        country = fname.split("_")[0]

        # year = last 4 consecutive digits before non-digit/end
        year_match = re.search(r"(\d{4})(?=\D*$)", fname)
        year = year_match.group(1) if year_match else None

        return country, year


    def build_dict(raster_list):
        """
        Build dict indexed by (country, year):
            { (country, year) : filepath }
        """
        d = {}
        for r in raster_list:
            country, year = extract_country_and_year(r)
            if country and year:
                d.setdefault((country, year), []).append(r)   # append many matches
        return d


    # Build lookup tables
    data_dict = build_dict(data_rasters)
    lc_dict = build_dict(landcover_rasters)

    # Find matching (country, year) keys
    common_keys = sorted(set(data_dict.keys()) & set(lc_dict.keys()))

    # Prepare output pairs
    # matched_pairs = [(data_dict[key], lc_dict[key]) for key in common_keys]
    
    matched_pairs = []

    for key in common_keys:
        for data_file, lc_file in itertools.product(data_dict[key], lc_dict[key]):
            matched_pairs.append((data_file, lc_file))   # <- flat tuple

    return matched_pairs

def align_raster_to_reference(ref_raster, input_raster, output_raster):
    """
    Resamples/reprojects input_raster to match CRS, resolution, extent of ref_raster.
    Uses nearest neighbor resampling for categorical data.
    """
    if not os.path.exists(os.path.dirname(output_raster)):
        os.makedirs(os.path.dirname(output_raster), exist_ok=True)

    ref_ds = gdal.Open(ref_raster)
    if ref_ds is None:
        raise FileNotFoundError(f"Reference raster not found: {ref_raster}")

    options = gdal.WarpOptions(
        format='GTiff',
        dstSRS=ref_ds.GetProjection(),
        width=ref_ds.RasterXSize,
        height=ref_ds.RasterYSize,
        resampleAlg='nearest', # It is a landcover dataset, so nearest neighbor resampling is appropriate
        outputBounds=[
            ref_ds.GetGeoTransform()[0],  # minX
            ref_ds.GetGeoTransform()[3] + ref_ds.RasterYSize * ref_ds.GetGeoTransform()[5],  # minY
            ref_ds.GetGeoTransform()[0] + ref_ds.RasterXSize * ref_ds.GetGeoTransform()[1],  # maxX
            ref_ds.GetGeoTransform()[3]  # maxY
        ],
        # Lets see if this works
        creationOptions=[
            "COMPRESS=DEFLATE",  
            "TILED=YES"          
    ]
    )

    result = gdal.Warp(destNameOrDestDS=output_raster,
                       srcDSOrSrcDSTab=input_raster,
                       options=options)

    if result is None:
        raise RuntimeError(f"gdal.Warp failed to generate: {output_raster}")

    result = None
    return output_raster

def landcover_mask_rasters(data_rasters, landcover_rasters, class_values, output_dir):
    """
    Masks each data raster by landcover classes.
    Automatically aligns landcover raster if shapes differ.
    """
    # Create the output dir
    os.makedirs(output_dir, exist_ok=True)

    # Match rasters by country + year
    # pairs = match_rasters_by_country_year(data_rasters, landcover_rasters)
    # pairs = match_rasters_by_country(data_rasters, landcover_rasters)
    pairs = match_all_rasters(data_rasters, landcover_rasters)

    if not pairs:
        print("No matching raster pairs found. Check filenames!")
        return

    for data_file, lc_file in pairs[:]: #ACHTUNG: Update this part
        print(f"\nProcessing pair:\n  DATA: {data_file}\n  LC  : {lc_file}")

        # Open data raster
        data_ds = gdal.Open(data_file)
        if data_ds is None:
            print(f"Cannot open data raster: {data_file}, skipping...")
            continue
        
        band = data_ds.GetRasterBand(1)
        data_arr = band.ReadAsArray()

        # original nodata
        nodata_value = band.GetNoDataValue()

        # Getnodatvalue requires always something
        if nodata_value is None:
            nodata_value = 0

        # Get the raster properties        
        geotrans = data_ds.GetGeoTransform()
        proj = data_ds.GetProjection()
        xsize = data_ds.RasterXSize
        ysize = data_ds.RasterYSize
        base_name = os.path.splitext(os.path.basename(data_file))[0]

        # Open landcover raster
        lc_ds = gdal.Open(lc_file)
        if lc_ds is None:
            print(f"Cannot open landcover raster: {lc_file}, skipping...")
            continue
        lc_arr = lc_ds.GetRasterBand(1).ReadAsArray()

        # Check shapes of raster and lc if an alignment is needed
        if lc_arr.shape != data_arr.shape:
            print("  - Landcover raster shape mismatch. Aligning...")
            aligned_lc_path = os.path.join(output_dir, "tmp_aligned_lc.tif")
            
            # The first variable is the reference
            align_raster_to_reference(data_file, lc_file, aligned_lc_path)
            
            lc_ds = gdal.Open(aligned_lc_path)
            lc_arr = lc_ds.GetRasterBand(1).ReadAsArray()
            print("  - Alignment done.")
        
        # Determine landcover nodata
        lc_nodata = lc_ds.GetRasterBand(1).GetNoDataValue()
        
        # ------------------------------------------
        # CASE 1 — class_values is None → mask ALL presence
        # ------------------------------------------
        if class_values is None:
            print("Masking all landcover presence (class_values=None)")

            # mask = all valid lc pixels (anything that is not nodata)
            mask = (lc_arr != lc_nodata)
            out_arr = np.where(mask, data_arr, nodata_value)
            # Use the lc name end.

            lc_name = os.path.splitext(os.path.basename(lc_file))[0]
            out_name = f"{base_name}_{lc_name.split('_')[-1]}.tif" # Get the second last part

            
            out_path = os.path.join(output_dir, out_name)

            driver = gdal.GetDriverByName("GTiff")
            out_ds = driver.Create(out_path, xsize, ysize, 1, gdal.GDT_Float32,
                                options=["COMPRESS=DEFLATE", "TILED=YES"])
            out_ds.SetGeoTransform(geotrans)
            out_ds.SetProjection(proj)
            band = out_ds.GetRasterBand(1)
            band.WriteArray(out_arr)
            band.SetNoDataValue(nodata_value)
            band.FlushCache()
            out_ds = None

        # ------------------------------------------
        # CASE 2 — class_values provided → existing behavior
        # ------------------------------------------
        else:    
            # Apply mask per class
            for lc_val in class_values:
                print(f"  - Masking class {lc_val}", end="")
                mask = (lc_arr == lc_val)
                out_arr = np.where(mask, data_arr, nodata_value)

                out_name = f"{base_name}_{lc_val}.tif"
                out_path = os.path.join(output_dir, out_name)

                driver = gdal.GetDriverByName("GTiff")
                out_ds = driver.Create(out_path, xsize, ysize, 1, gdal.GDT_Float32,
                                    options=["COMPRESS=DEFLATE", "TILED=YES"])
                out_ds.SetGeoTransform(geotrans)
                out_ds.SetProjection(proj)
                band = out_ds.GetRasterBand(1)
                band.WriteArray(out_arr)
                band.SetNoDataValue(nodata_value)
                band.FlushCache()
                out_ds = None


            
        print(f"✓ Finished processing {base_name}")
        # Clean the variable
        # Close ALL GDAL datasets
        try:
            del lc_ds
            # delete temp file safely
            os.remove(aligned_lc_path)
        except: 
            pass

        
        
    print("\nAll rasters processed successfully.")

def landcover_mask_rasters_in_blocks(data_rasters, landcover_rasters, class_values, output_dir, output_files_list, block_size=1024):
    """
    Masks each data raster by landcover classes using block processing.
    Avoids loading entire rasters into memory.
    """

    os.makedirs(output_dir, exist_ok=True)

    pairs = match_all_rasters(data_rasters, landcover_rasters)

    # pairs = filter_already_processed(
    # pairs,
    # output_files_list,
    # output_dir
    # )



    if not pairs:
        print("No matching raster pairs found. Check filenames!")
        return

    for data_file, lc_file in pairs[:]:
        print(f"\nProcessing pair:\n  DATA: {data_file}\n  LC  : {lc_file}")

        data_ds = gdal.Open(data_file)
        lc_ds = gdal.Open(lc_file)

        if data_ds is None or lc_ds is None:
            print("Error opening rasters, skipping...")
            continue

        data_band = data_ds.GetRasterBand(1)
        lc_band = lc_ds.GetRasterBand(1)

        nodata_value = data_band.GetNoDataValue()
        if nodata_value is None:
            nodata_value = 0

        lc_nodata = lc_band.GetNoDataValue()

        geotrans = data_ds.GetGeoTransform()
        proj = data_ds.GetProjection()
        xsize = data_ds.RasterXSize
        ysize = data_ds.RasterYSize

        base_name = os.path.splitext(os.path.basename(data_file))[0]

        # ------------------------------------------
        # ALIGN LANDCOVER IF NEEDED
        # ------------------------------------------
        if (lc_ds.RasterXSize != xsize) or (lc_ds.RasterYSize != ysize):
            print("  - Aligning landcover raster...")

            aligned_lc_path = os.path.join(output_dir, os.path.basename(lc_file).replace(".tif", "") + "_aligned.tif")
            # we don't want to align this several times
            if not os.path.exists(aligned_lc_path):
                align_raster_to_reference(data_file, lc_file, aligned_lc_path)

            lc_ds = gdal.Open(aligned_lc_path)
            lc_band = lc_ds.GetRasterBand(1)
            lc_nodata = lc_band.GetNoDataValue()
        else:
            print("Data is already aligned")

        driver = gdal.GetDriverByName("GTiff")
        # ------------------------------------------
        # CASE 1: ALL LANDCOVER (class_values=None)
        # ------------------------------------------
        if class_values is None:
            lc_name = os.path.splitext(os.path.basename(lc_file))[0]
            out_name = f"{base_name}_{lc_name.split('_')[-1]}.tif"
            out_path = os.path.join(output_dir, out_name)

            out_ds = driver.Create(
                out_path, xsize, ysize, 1, gdal.GDT_Float32,
                options=["COMPRESS=DEFLATE", "TILED=YES"]
            )
            out_ds.SetGeoTransform(geotrans)
            out_ds.SetProjection(proj)

            out_band = out_ds.GetRasterBand(1)
            out_band.SetNoDataValue(nodata_value)

            # BLOCK LOOP
            for y in range(0, ysize, block_size):
                rows = min(block_size, ysize - y)

                for x in range(0, xsize, block_size):
                    cols = min(block_size, xsize - x)

                    print(f"Processing block: row={y}:{y+rows}, col={x}:{x+cols}", end="\r")

                    data_block = data_band.ReadAsArray(x, y, cols, rows)
                    lc_block = lc_band.ReadAsArray(x, y, cols, rows)

                    # Skip useless blocks
                    if np.all(data_block == nodata_value):
                        continue
                    if lc_nodata is not None and np.all(lc_block == lc_nodata):
                        continue

                    mask = (lc_block != lc_nodata)
                    out_block = np.where(mask, data_block, nodata_value)

                    out_band.WriteArray(out_block, x, y)

            out_ds = None

        # ------------------------------------------
        # CASE 2: SPECIFIC CLASSES
        # ------------------------------------------
        else:
            out_datasets = {}
            out_bands = {}

            for lc_val in class_values:
                out_name = f"{base_name}_{lc_val}.tif"
                out_path = os.path.join(output_dir, out_name)

                ds = driver.Create(
                    out_path, xsize, ysize, 1, gdal.GDT_Float32,
                    options=["COMPRESS=DEFLATE", "TILED=YES"]
                )
                ds.SetGeoTransform(geotrans)
                ds.SetProjection(proj)

                band = ds.GetRasterBand(1)
                band.SetNoDataValue(nodata_value)

                out_datasets[lc_val] = ds
                out_bands[lc_val] = band

            # BLOCK LOOP
            for y in range(0, ysize, block_size):
                rows = min(block_size, ysize - y)

                for x in range(0, xsize, block_size):
                    cols = min(block_size, xsize - x)

                    data_block = data_band.ReadAsArray(x, y, cols, rows)
                    lc_block = lc_band.ReadAsArray(x, y, cols, rows)

                    # Skip useless blocks
                    if np.all(data_block == nodata_value):
                        continue
                    if lc_nodata is not None and np.all(lc_block == lc_nodata):
                        continue

                    for lc_val in class_values:
                        mask = (lc_block == lc_val)

                        # Skip if class not present
                        if not np.any(mask):
                            continue

                        out_block = np.where(mask, data_block, nodata_value)
                        out_bands[lc_val].WriteArray(out_block, x, y)

            # Close datasets
            for ds in out_datasets.values():
                ds = None

        print(f"✓ Finished processing {base_name}")

        # Cleanup
        # try:
        #     if 'aligned_lc_path' in locals():
        #         os.remove(aligned_lc_path)
        # except:
        #     pass

        data_ds = None
        lc_ds = None

    print("\nAll rasters processed successfully.")

def build_output_path(p1, p2, output_dir):
    """
    Build expected output filename from input pair.
    """

    base = os.path.basename(p1).replace(".tif", "")

    # Extract vulnerability type from mask filename
    if "highvulnerability" in p2:
        suffix = "highvulnerability"
    elif "mediumvulnerability" in p2:
        suffix = "mediumvulnerability"
    elif "lowvulnerability" in p2:
        suffix = "lowvulnerability"
    elif "unknownvulnerability" in p2:
        suffix = "unknownvulnerability"
    else:
        raise ValueError(f"Unknown vulnerability in: {p2}")

    return os.path.join(output_dir, f"{base}_{suffix}.tif")

def filter_already_processed(file_pairs, output_files_list, output_dir):
    """
    Remove pairs whose expected output already exists.
    """

    existing_outputs = set(output_files_list)

    filtered = []

    for p1, p2 in file_pairs:
        expected_output = build_output_path(p1, p2, output_dir)

        if expected_output not in existing_outputs:
            filtered.append((p1, p2))
        else:
            print(f"Skipping (already processed): {expected_output}")

    return filtered


In [12]:
"""Inputs for classifier."""
# landcover_classes_csv = r"Y:\z_resources\un_gbf\01_aggregation_phase\wetland_classes.csv"
landcover_classes_csv = None

raster_files_path = r"Y:\z_resources\un_gbf\01_aggregation_phase\03_masked_outputs\coastal_protection_01"

landcover_files_path = r"Y:\z_resources\un_gbf\01_aggregation_phase\02_mask_rasters\coastal_protection\00_current_processing_2"

output_dir = r"Y:\z_resources\un_gbf\01_aggregation_phase\03_masked_outputs\coastal_protection_03"

In [13]:
raster_files_list = get_raster_file_list(raster_files_path)
landcover_files_list = get_raster_file_list(landcover_files_path)

lc_classes_list = read_csv_first_column(landcover_classes_csv)

output_files_list = get_raster_file_list(output_dir)

In [14]:
file_pairs = match_all_rasters(raster_files_list, landcover_files_list)
print(len(file_pairs), file_pairs[0:1])

16 [('Y:\\z_resources\\un_gbf\\01_aggregation_phase\\03_masked_outputs\\coastal_protection_01\\cp_demand_merged_total_1.tif', 'Y:\\z_resources\\un_gbf\\01_aggregation_phase\\02_mask_rasters\\coastal_protection\\00_current_processing_2\\global_nasa_grdiv1_sociodemographic_epsg4326_1km_2010_2020_v1_unknownvulnerability.tif')]


In [ ]:
filtered_file_pairs = filter_already_processed(
    file_pairs,
    output_files_list,
    output_dir
)

print(f"Remaining pairs: {len(filtered_file_pairs)}")

In [ ]:
country_pairs = match_rasters_by_country(raster_files_list, landcover_files_list)
country_pairs[:]

In [ ]:
file_pairs = match_rasters_by_country_year(raster_files_list, landcover_files_list)
file_pairs[:]

In [16]:
# classes = read_landcover_classes(csv_path)
landcover_mask_rasters_in_blocks(raster_files_list, landcover_files_list, lc_classes_list, output_dir, output_files_list, block_size=1024)


Processing pair:
  DATA: Y:\z_resources\un_gbf\01_aggregation_phase\03_masked_outputs\coastal_protection_01\cp_demand_merged_total_1.tif
  LC  : Y:\z_resources\un_gbf\01_aggregation_phase\02_mask_rasters\coastal_protection\00_current_processing_2\global_nasa_grdiv1_sociodemographic_epsg4326_1km_2010_2020_v1_unknownvulnerability.tif
  - Aligning landcover raster...
✓ Finished processing cp_demand_merged_total_12:720001

Processing pair:
  DATA: Y:\z_resources\un_gbf\01_aggregation_phase\03_masked_outputs\coastal_protection_01\cp_demand_merged_total_183.tif
  LC  : Y:\z_resources\un_gbf\01_aggregation_phase\02_mask_rasters\coastal_protection\00_current_processing_2\global_nasa_grdiv1_sociodemographic_epsg4326_1km_2010_2020_v1_unknownvulnerability.tif
  - Aligning landcover raster...
✓ Finished processing cp_demand_merged_total_183720001

Processing pair:
  DATA: Y:\z_resources\un_gbf\01_aggregation_phase\03_masked_outputs\coastal_protection_01\cp_demand_merged_total_185.tif
  LC  : Y:\z